# Vision & Grounding Models Testing SuiteThis notebook tests specialized OCR and grounding models on Kaggle: **Chandra 2**, **LocateAnything**, **MiniCPM-V 2.6**, **PaliGemma**, **GOT-OCR**, **GLM-OCR**, and **PaddleOCR**.

In [ ]:
"""
================================================================================
KAGGLE CELL 1: Environment Setup & Installations
Note: We explicitly DO NOT upgrade PyTorch to prevent CUDA driver mismatch errors on Kaggle T4s.
Note: We DO NOT upgrade Pillow or huggingface_hub to prevent ImportErrors.
================================================================================
"""
!pip install transformers accelerate bitsandbytes timm einops verovio --upgrade
# PaddleOCR: pin compatible versions to avoid set_optimization_level errors
!pip install paddlepaddle-gpu==3.0.0 paddleocr==3.0.2

In [ ]:
"""
================================================================================
KAGGLE CELL 2: Hugging Face Authentication
Many of these models (PaliGemma, MiniCPM) are gated. You MUST accept their 
license on huggingface.co and log in here using a token with read access.
================================================================================
"""
from huggingface_hub import login
import os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Successfully logged in via Kaggle Secrets!")
except:
    print("Could not find HF_TOKEN in Kaggle Secrets.")
    print("Please log in manually:")
    login()

In [ ]:
import torch
import gc
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, AutoModel, AutoTokenizer

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM Cleared.")

TEST_IMAGE_PATH = "/kaggle/input/datasets/crypticsaviour/uhm123/log book.png"
image = Image.open(TEST_IMAGE_PATH).convert("RGB")

In [ ]:
"""
================================================================================
TEST 1: Chandra 2 (datalab-to/chandra-ocr-2)
================================================================================
"""
def test_chandra_2():
    print("\n--- Loading Chandra OCR 2 ---")
    model_id = "datalab-to/chandra-ocr-2"
    
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, 
        device_map="auto", 
        torch_dtype=torch.float16, 
        trust_remote_code=True
    ).eval()
    
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Extract all text and structure:"}
            ]
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=image, padding=True, return_tensors="pt").to("cuda", torch.float16)
    
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=512)
        
    print("Chandra 2 Output:\n", processor.decode(outputs[0], skip_special_tokens=True))
    
    del model, processor, inputs
    clear_vram()

In [ ]:
"""
================================================================================
TEST 2: LocateAnything (nvidia/LocateAnything-3B)
================================================================================
"""
def test_locate_anything(target_object="handwritten text"):
    print("\n--- Loading NVIDIA LocateAnything ---")
    model_id = "nvidia/LocateAnything-3B"
    
    import transformers
    
    old_init = transformers.modeling_utils.PreTrainedModel.__init__
    def patched_init(self, config, *inputs, **kwargs):
        target_cls = None
        for base in self.__class__.__mro__:
            if '_check_and_adjust_attn_implementation' in base.__dict__:
                target_cls = base
                break
                
        if target_cls and target_cls.__name__ != 'PreTrainedModel':
            old_check = target_cls.__dict__['_check_and_adjust_attn_implementation']
            @classmethod
            def safe_check(cls, *args, **kw):
                kw.pop('allow_all_kernels', None)
                if isinstance(old_check, classmethod):
                    return old_check.__func__(cls, *args, **kw)
                else:
                    return old_check(cls, *args, **kw)
            setattr(target_cls, '_check_and_adjust_attn_implementation', safe_check)
            try:
                old_init(self, config, *inputs, **kwargs)
            finally:
                setattr(target_cls, '_check_and_adjust_attn_implementation', old_check)
        else:
            old_init(self, config, *inputs, **kwargs)
            
    transformers.modeling_utils.PreTrainedModel.__init__ = patched_init
    
    if not hasattr(transformers.models.qwen2.configuration_qwen2.Qwen2Config, "rope_theta"):
        transformers.models.qwen2.configuration_qwen2.Qwen2Config.rope_theta = 1000000.0
    
    try:
        model = AutoModel.from_pretrained(
            model_id, 
            trust_remote_code=True, 
            device_map="auto", 
            torch_dtype=torch.float16
        ).eval()
        
        # FIX: LocateAnything uses standard Qwen-VL style processing, NOT model.chat()!
        processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"Locate: {target_object}"}
                ]
            }
        ]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=image, padding=True, return_tensors="pt").to("cuda", torch.float16)
        
        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=256)
            
        res = processor.decode(outputs[0], skip_special_tokens=True)
        print(f"LocateAnything Output for '{target_object}':\n", res)
        
        del model, processor, inputs
    except Exception as e:
        print(f"Error running LocateAnything inference: {e}")
    finally:
        transformers.modeling_utils.PreTrainedModel.__init__ = old_init
        clear_vram()

In [ ]:
"""
================================================================================
TEST 3: MiniCPM-V 2.6 (openbmb/MiniCPM-V-2_6)
================================================================================
"""
def test_minicpm():
    import torch.nn as nn
    
    print("\n--- Loading MiniCPM-V 2.6 ---")
    model_id = "openbmb/MiniCPM-V-2_6"
    
    old_getattr = nn.Module.__getattr__
    def safe_getattr(self, name):
        if name == 'all_tied_weights_keys':
            return {}
        return old_getattr(self, name)
    nn.Module.__getattr__ = safe_getattr

    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_4bit=True)
    
    model = AutoModel.from_pretrained(
        model_id, 
        trust_remote_code=True, 
        device_map={"": 0},  
        quantization_config=quantization_config,
        torch_dtype=torch.float16
    ).eval()
    
    nn.Module.__getattr__ = old_getattr
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    
    msgs = [{'role': 'user', 'content': 'Extract all text verbatim.'}]
    
    res = model.chat(
        image=image,
        msgs=msgs,
        tokenizer=tokenizer,
        sampling=True, 
        temperature=0.7
    )
    
    print("MiniCPM Output:\n", res)
    
    del model, tokenizer
    clear_vram()

In [ ]:
"""
================================================================================
TEST 4: PaliGemma 3B (google/paligemma-3b-pt-224)
================================================================================
"""
def test_paligemma():
    print("\n--- Loading PaliGemma 3B ---")
    from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor
    model_id = "google/paligemma-3b-pt-224"
    
    model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto").eval()
    processor = PaliGemmaProcessor.from_pretrained(model_id)
    
    prompt = "ocr"  # PaliGemma relies on specific short task prompts like 'ocr' or 'caption'
    model_inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda", torch.float16)
    
    with torch.inference_mode():
        generation = model.generate(**model_inputs, max_new_tokens=256)
        # Remove prompt tokens from output
        generation = generation[0][model_inputs['input_ids'].shape[-1]:]
        decoded = processor.decode(generation, skip_special_tokens=True)
        
    print("PaliGemma Output:\n", decoded)
    
    del model, processor, model_inputs
    clear_vram()

In [ ]:
"""
================================================================================
TEST 5: GOT-OCR (stepfun-ai/GOT-OCR-2.0-hf)
Uses HF-native AutoModelForImageTextToText. This model has NO chat template,
so we pass the image directly to the processor and generate.
================================================================================
"""
def test_got_ocr():
    print("\n--- Loading GOT-OCR 2.0 (HF-native) ---")
    import warnings
    warnings.filterwarnings("ignore", category=SyntaxWarning)
    
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model_id = "stepfun-ai/GOT-OCR-2.0-hf"
    
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        trust_remote_code=True,
        device_map="auto",
        torch_dtype=torch.float16,
        attn_implementation="eager",  # Avoid SDPA tensor shape mismatch
    ).eval()
    
    # GOT-OCR-2.0-hf has NO chat template. Pass image directly to processor.
    inputs = processor(images=image, return_tensors="pt").to("cuda")
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            tokenizer=processor.tokenizer,
            stop_strings="<|im_end|>",
            max_new_tokens=4096,
        )
    
    prompt_len = inputs["input_ids"].shape[-1]
    result = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    
    print("GOT-OCR Output:\n", result)
    
    del model, processor, inputs
    clear_vram()

In [ ]:
"""
================================================================================
TEST 6: GLM-OCR (zai-org/GLM-OCR)
================================================================================
"""
def test_glm_ocr():
    print("\n--- Loading GLM-OCR ---")
    from transformers import AutoProcessor, GlmOcrForConditionalGeneration
    model_id = "zai-org/GLM-OCR"
    
    processor = AutoProcessor.from_pretrained(model_id)
    model = GlmOcrForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
    ).to("cuda").eval()
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Extract all text from this image."}
            ],
        }
    ]
    
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")
    
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=2048)
    
    result = processor.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    
    print("GLM-OCR Output:\n", result)
    
    del model, processor, inputs
    clear_vram()

In [ ]:
"""
================================================================================
TEST 7: PaddleOCR
Monkey-patches set_optimization_level if missing (PaddlePaddle version mismatch),
then tries PaddleOCR-VL, then classic PaddleOCR as fallback.
================================================================================
"""
def test_paddle_ocr():
    print("\n--- Loading PaddleOCR ---")
    import glob, shutil, os
    
    # ── Monkey-patch for PaddlePaddle version mismatch ──
    # Newer paddleocr/paddlex calls config.set_optimization_level() which
    # doesn't exist in some paddlepaddle-gpu builds on Kaggle.
    try:
        import paddle.inference as paddle_infer
        _orig_config = paddle_infer.Config
        class PatchedConfig(_orig_config):
            def set_optimization_level(self, level):
                # silently skip if the underlying C++ API doesn't support it
                if hasattr(super(), 'set_optimization_level'):
                    super().set_optimization_level(level)
        paddle_infer.Config = PatchedConfig
        print("Applied set_optimization_level compatibility patch.")
    except Exception:
        pass
    
    # ── Try PaddleOCR-VL first ──
    try:
        from paddleocr import PaddleOCRVL
        try:
            pipeline = PaddleOCRVL(pipeline_version="v1.6")
        except Exception:
            pipeline = PaddleOCRVL(pipeline_version="v1.5")
        
        output = pipeline.predict(TEST_IMAGE_PATH)
        
        if output:
            res = output[0]
            extracted_text = ""
            if hasattr(res, "markdown"):
                extracted_text = res.markdown
            elif isinstance(res, dict) and "markdown" in res:
                extracted_text = res["markdown"]
            
            if not extracted_text:
                temp_dir = "/kaggle/working/temp_pocr_out"
                os.makedirs(temp_dir, exist_ok=True)
                try:
                    res.save_to_markdown(temp_dir)
                    md_files = glob.glob(os.path.join(temp_dir, "*.md"))
                    if md_files:
                        with open(md_files[0], "r", encoding="utf-8") as f:
                            extracted_text = f.read().strip()
                finally:
                    shutil.rmtree(temp_dir, ignore_errors=True)
            
            print("PaddleOCR-VL Output:\n", extracted_text)
        else:
            print("PaddleOCR-VL returned no output.")
        return
    except Exception as e:
        print(f"PaddleOCR-VL unavailable ({e}), trying classic PaddleOCR...")
    
    # ── Fallback: Classic PaddleOCR ──
    try:
        from paddleocr import PaddleOCR
        ocr = PaddleOCR(use_angle_cls=True, lang="en")
        result = ocr.ocr(TEST_IMAGE_PATH, cls=True)
        print("PaddleOCR Output:")
        if result and result[0]:
            for line in result[0]:
                print(line[1][0])
    except Exception as e:
        print(f"Classic PaddleOCR also failed: {e}")
        print("Try: !pip install paddlepaddle-gpu==3.0.0 paddleocr==3.0.2")

In [ ]:
# test_chandra_2()

In [ ]:
# test_locate_anything("handwritten text")

In [ ]:
# test_minicpm()

In [ ]:
# test_paligemma()

In [ ]:
# test_got_ocr()

In [ ]:
# test_glm_ocr()

In [ ]:
# test_paddle_ocr()